# 🏛️ Multi-View 3D Reconstruction & Multi-Representation Modeling
### *Google Colab Cloud Execution Environment (Free NVIDIA GPU)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anjan-Reddy/multiview_3d_cv_project/blob/main/3D_Reconstruction_Colab.ipynb)

This notebook allows you to execute the complete end-to-end 3D reconstruction pipeline on Google Colab using a free cloud NVIDIA GPU.

**Reconstruction Architecture:**
1. **Neural Preprocessing**: `rembg` automated U$^2$-Net foreground segmentation.
2. **Sparse SfM**: SIFT feature extraction, epipolar RANSAC, and bundle adjustment via COLMAP.
3. **Dense MVS**: CUDA PatchMatch stereo depth and normal estimation.
4. **Multi-Representation Export**: Screened Poisson surface meshing and Open3D voxel occupancy grids.
5. **Interactive 3D Visualization**: Orbit, inspect, and download generated `.ply` models.

## ⚡ Step 1: Check GPU Acceleration
Ensure the Colab runtime has a GPU allocated (**Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**).

In [ ]:
!nvidia-smi

## 📦 Step 2: Install COLMAP & System Dependencies
Install standalone COLMAP photogrammetry engine and OpenGL virtual framebuffer for headless rendering.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq colmap libgl1-mesa-glx xvfb
!colmap -h | head -n 5

## 📥 Step 3: Install Python Dependencies
Install Open3D, rembg, OpenCV, Plotly, and related 3D vision libraries.

In [ ]:
!pip install -q open3d opencv-python-headless pillow "rembg[cpu]>=2.0.50" flask scipy matplotlib tqdm plotly

## 📂 Step 4: Clone or Mount Repository
If running inside Google Colab, clone the GitHub repository.

In [ ]:
import os, sys
from pathlib import Path

# If src folder is not in current working dir, clone repository
if not os.path.exists("src"):
    repo_url = "https://github.com/Anjan-Reddy/multiview_3d_cv_project.git"
    !git clone {repo_url} repo
    %cd repo

print("Working directory:", os.getcwd())
assert os.path.exists("src"), "Error: src folder not found!"

## 📸 Step 5: Select Dataset or Upload Custom Photographs
Choose one of the bundled datasets (Owl, South Building, Dog, Apple), or upload your own photos into `inputs/<custom_dataset_name>/`.

In [ ]:
#@title Select Dataset or Custom Upload
dataset_choice = "Owl" #@param ["Owl", "South Building", "Dog", "Apple", "Upload Custom Images"]

if dataset_choice == "Owl":
    images_dir = "inputs/images/owl-images"
    use_masking = True
    dataset_id = "owl"
elif dataset_choice == "South Building":
    images_dir = "inputs/images/Building"
    use_masking = False
    dataset_id = "Building"
elif dataset_choice == "Dog":
    images_dir = "inputs/images/Dog_RGB"
    use_masking = True
    dataset_id = "dog"
elif dataset_choice == "Apple":
    images_dir = "inputs/images/Apple"
    use_masking = True
    dataset_id = "Apple"
else:
    from google.colab import files
    custom_name = input("Enter dataset name (e.g. my_object): ").strip() or "custom_object"
    upload_dir = Path(f"inputs/images/{custom_name}")
    upload_dir.mkdir(parents=True, exist_ok=True)
    print(f"Upload photographs for {custom_name}...")
    uploaded = files.upload()
    for fn, data in uploaded.items():
        with open(upload_dir / fn, "wb") as f:
            f.write(data)
    images_dir = str(upload_dir)
    use_masking = True
    dataset_id = custom_name

print(f"\nSelected Dataset: {dataset_id}")
print(f"Image Directory:  {images_dir}")
print(f"Use Masking:      {use_masking}")

## ✂️ Step 6: Neural Foreground Preprocessing (rembg)
For turntable/tabletop objects, `rembg` segments the foreground and suppresses background clutter during SIFT feature matching.

In [ ]:
if use_masking:
    prepared_dir = f"prepared/{dataset_id}"
    !python -m src.preprocess_images --images "{images_dir}" --output "{prepared_dir}" --max-image-size 1000
    final_images = f"{prepared_dir}/images"
    final_masks = f"{prepared_dir}/masks"
else:
    final_images = images_dir
    final_masks = None

print("Preprocessing ready. Final images:", final_images)

## 🚀 Step 7: Run End-to-End 3D Reconstruction Pipeline
Executes COLMAP feature extraction, exhaustive matching, incremental SfM, CUDA PatchMatch MVS, Screened Poisson meshing, and Open3D voxelization.

In [ ]:
output_dir = f"outputs/{dataset_id}"

cmd = [
    "python", "-m", "src.pipeline",
    "--images", final_images,
    "--output", output_dir,
    "--colmap", "colmap",
    "--gpu", "on",
    "--matcher", "exhaustive",
    "--max-image-size", "1000",
    "--overwrite"
]
if final_masks:
    cmd.extend(["--masks", final_masks])

!{" ".join(cmd)}

## 📊 Step 8: View Metrics & Evaluation Summary

In [ ]:
import json
metrics_path = Path(output_dir) / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("=" * 40)
    print(f"3D RECONSTRUCTION RESULTS: {dataset_id.upper()}")
    print("=" * 40)
    for k, v in metrics.items():
        print(f"{k:25s}: {v}")
    print("=" * 40)
else:
    print("No metrics.json found at", metrics_path)

## 🌐 Step 9: Interactive 3D Visualization in Colab
Inspect the reconstructed point cloud directly in your browser using Plotly 3D scatter rendering.

In [ ]:
import open3d as o3d
import numpy as np
import plotly.graph_objects as go

pcd_file = Path(output_dir) / "point_cloud.ply"
if pcd_file.exists():
    pcd = o3d.io.read_point_cloud(str(pcd_file))
    # Downsample points for fast web rendering in Colab
    pcd_down = pcd.voxel_down_sample(voxel_size=0.02) if len(pcd.points) > 50000 else pcd
    pts = np.asarray(pcd_down.points)
    colors = np.asarray(pcd_down.colors)

    color_strings = [f"rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})" for c in colors] if len(colors) == len(pts) else "deepskyblue"

    fig = go.Figure(data=[go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=dict(size=2, color=color_strings, opacity=0.8)
    )])
    fig.update_layout(
        title=f"Interactive 3D Point Cloud: {dataset_id}",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="rgb(15, 23, 42)"
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        height=650
    )
    fig.show()
else:
    print("Point cloud not found at:", pcd_file)

## 💾 Step 10: Download Generated 3D Models
Package deliverables (`point_cloud.ply`, `mesh.ply`, `voxels.npz`, `metrics.json`) into a ZIP archive and download to your local machine.

In [ ]:
import shutil
from google.colab import files

zip_filename = f"{dataset_id}_3d_reconstruction.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), "zip", output_dir)
print(f"Created archive: {zip_filename}")
files.download(zip_filename)